<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-DeepLearning/blob/main/W06_Transfer_Learning_Loesungsvorschlag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Woche 6 – Transfer Learning: Von Grund auf, Feature Extraction oder Finetuning?

**Lösungsversion** mit Musterlösungen und Outputs eines Testlaufs.

In diesem Notebook vergleichst du drei Strategien, um ein Bildklassifikationsmodell für eine neue Aufgabe zu trainieren:

| Strategie | Startgewichte | Was wird trainiert? |
|---|---|---|
| **Von Grund auf** (*from scratch*) | zufällig | alles |
| **Feature Extraction** | ImageNet | nur der neue Kopf (Head). Alternativ könnte mit den Features aus dem Kopf auch ein anderes ML-Modell verwendet werden. |
| **Finetuning** | ImageNet | alles, Backbone mit kleinerer Lernrate |

Jede Strategie trainierst du einmal mit **100 %** und einmal mit **10 %** der Trainingsdaten.

**Anwendungsfall:** Klassifikation von Satellitenbildern in 10 Landnutzungsklassen (EuroSAT). So etwas brauchen zum Beispiel Versicherungen, Agrarbetriebe oder Raumplanungsämter.

**Aufbau:** Das Notebook folgt dem Vierschritt aus dem Projekt-Baukasten: *Fragestellung → Experiment-Design → Resultate → Interpretation*.

**Vorbereitung:** *Laufzeit → Laufzeittyp ändern → T4 GPU*. Auf der GPU sollten alle Experimente zusammen nur wenige Minuten dauern, auf einer CPU rund 20 Minuten.

## 0 Setup

In [ ]:
import copy
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.datasets import EuroSAT
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.transforms.functional import pil_to_tensor

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device("cpu")
print("PyTorch", torch.__version__, "| Gerät:", device)

## 1 Daten: EuroSAT

EuroSAT enthält 27'000 Satellitenbilder (Sentinel-2, RGB, 64×64 Pixel) in 10 Klassen. Quelle: Helber et al. (2019), *EuroSAT: A Novel Dataset and Deep Learning Benchmark for Land Use and Land Cover Classification*.

Damit die Experimente in der Lektion Platz haben, arbeiten wir mit einer **kleinen, stratifizierten Stichprobe**:

| Split | Bilder pro Klasse | Total |
|---|---|---|
| Training 100 % | 100 | 1'000 |
| Training 10 % | 10 (Teilmenge davon) | 100 |
| Validierung | 30 | 300 |
| Test | 100 | 1'000 |

Training, Validierung und Test sind **disjunkt**. Der 10-%-Split ist eine Teilmenge des 100-%-Splits, damit der Vergleich nur die Datenmenge verändert.

In [ ]:
dataset = EuroSAT(root="data", download=True)
CLASSES = dataset.classes
NUM_CLASSES = len(CLASSES)
print(len(dataset), "Bilder,", NUM_CLASSES, "Klassen:", CLASSES)

N_TRAIN, N_VAL, N_TEST, N_SMALL = 100, 30, 100, 10

# Stratifizierte Stichprobe: pro Klasse mischen und aufteilen
rng = np.random.default_rng(SEED)
targets = np.array(dataset.targets)
split_idx = {"train": [], "train_10": [], "val": [], "test": []}
for c in range(NUM_CLASSES):
    idx = rng.permutation(np.where(targets == c)[0])
    split_idx["train"] += list(idx[:N_TRAIN])
    split_idx["train_10"] += list(idx[:N_SMALL])                    # Teilmenge von train
    split_idx["val"] += list(idx[N_TRAIN:N_TRAIN + N_VAL])
    split_idx["test"] += list(idx[N_TRAIN + N_VAL:N_TRAIN + N_VAL + N_TEST])

def load_split(indices):
    # Bilder als uint8-Tensoren (N, 3, 64, 64) in den Speicher laden
    x = torch.stack([pil_to_tensor(dataset[i][0]) for i in indices])
    y = torch.tensor([targets[i] for i in indices])
    return x, y

data = {name: load_split(idx) for name, idx in split_idx.items()}
for name, (x, y) in data.items():
    print(f"{name:9s} {tuple(x.shape)}  Bilder pro Klasse: {torch.bincount(y).tolist()}")

In [ ]:
x, y = data["train"]
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for c, ax in enumerate(axes.flat):
    i = (y == c).nonzero()[0].item()
    ax.imshow(x[i].permute(1, 2, 0))
    ax.set_title(CLASSES[c], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2 Vorverarbeitung passend zum vortrainierten Modell

Ein vortrainiertes Modell erwartet Inputs, die so aussehen wie seine Trainingsdaten. Das sind hier ImageNet-Fotos, die mit einem bestimmten **Mittelwert und einer Standardabweichung pro Farbkanal** normalisiert wurden.

Die Gewichte bringen ihre eigene Vorverarbeitung mit. Schau sie dir an:

In [ ]:
weights = ResNet18_Weights.DEFAULT
print(weights)
print(weights.transforms())

Die Standard-Vorverarbeitung skaliert auf 224×224 Pixel. Damit die Lektion schneller läuft, vergrössern wir die 64×64-Bilder nur auf **128×128 Pixel**. Die **Normalisierung** übernehmen wir aber genau so, wie sie beim Vortraining verwendet wurde.

### TODO A: Normalisierung
Trage die Werte für `MEAN` und `STD` aus der Ausgabe oben ein (je 3 Werte, einer pro Farbkanal R, G, B).

In [ ]:
IMG_SIZE = 128

# TODO A: Mittelwert und Standardabweichung aus weights.transforms()
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

In [ ]:
# Kontrollzelle A
if MEAN is None or STD is None:
    print("TODO A ist noch offen.")
else:
    assert np.allclose(MEAN, weights.transforms().mean) and np.allclose(STD, weights.transforms().std), \
        "Die Werte stimmen nicht mit weights.transforms() überein."
    print("TODO A ist korrekt.")

In [ ]:
def preprocess(x_uint8, normalize=True):
    # uint8 (N,3,64,64) -> float (N,3,IMG_SIZE,IMG_SIZE), normalisiert
    x = x_uint8.float() / 255.0
    x = F.interpolate(x, size=IMG_SIZE, mode="bilinear", antialias=True, align_corners=False)
    if normalize:
        mean = torch.tensor(MEAN, device=x.device).view(1, 3, 1, 1)
        std = torch.tensor(STD, device=x.device).view(1, 3, 1, 1)
        x = (x - mean) / std
    return x

def random_flip(x):
    # Leichte Data Augmentation: zufälliges horizontales und vertikales Spiegeln pro Bild
    # (bei Satellitenbildern gibt es kein «oben» und «unten»)
    n = x.shape[0]
    h = torch.rand(n, device=x.device) < 0.5
    v = torch.rand(n, device=x.device) < 0.5
    x = torch.where(h.view(-1, 1, 1, 1), x.flip(3), x)
    x = torch.where(v.view(-1, 1, 1, 1), x.flip(2), x)
    return x

## 3 Modell: Backbone und Kopf

Wir verwenden ein **ResNet-18**. Es besteht aus einem **Backbone** (Faltungsschichten, die Merkmale extrahieren) und einem **Kopf** `model.fc`, einer einzelnen linearen Schicht mit 1'000 Ausgängen für die ImageNet-Klassen.

In [ ]:
m = resnet18(weights=weights)
print("Kopf des vortrainierten Modells:", m.fc)
print("Hauptblöcke:", [name for name, _ in m.named_children()])
del m

### TODO B: Kopf ersetzen und Backbone einfrieren

Ergänze die Funktion `build_model`:

- **B1:** Bei `feature_extraction` sollen **alle** vorhandenen Parameter eingefroren werden (`requires_grad = False`).
- **B2:** Bei allen drei Strategien ersetzt du den Kopf durch eine neue lineare Schicht mit `NUM_CLASSES` Ausgängen.

*Tipp:* Die Anzahl Eingänge des alten Kopfes findest du in `model.fc.in_features`. Überlege dir, warum die Reihenfolge von B1 und B2 wichtig ist.

In [ ]:
STRATEGIES = ["scratch", "feature_extraction", "finetuning"]

def build_model(strategy):
    if strategy == "scratch":
        model = resnet18(weights=None)           # zufällige Startgewichte
    else:
        model = resnet18(weights=weights)        # ImageNet-Gewichte

    # TODO B1: bei Feature Extraction alle Parameter einfrieren
    if strategy == "feature_extraction":
        for p in model.parameters():
            p.requires_grad = False

    # TODO B2: neuen Kopf mit NUM_CLASSES Ausgängen einsetzen
    # (ein neues nn.Linear ist automatisch trainierbar, darum zuerst einfrieren, dann ersetzen)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

    return model.to(device)

In [ ]:
# Kontrollzelle B
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

try:
    for s in STRATEGIES:
        mdl = build_model(s)
        total, trainable = count_params(mdl)
        print(f"{s:20s} Parameter total: {total:>10,}  trainierbar: {trainable:>10,}  ({trainable / total:.2%})")
        assert mdl.fc.out_features == NUM_CLASSES, "Der Kopf hat nicht NUM_CLASSES Ausgänge."
        if s == "feature_extraction":
            assert trainable == mdl.fc.weight.numel() + mdl.fc.bias.numel(), \
                "Bei Feature Extraction soll nur der Kopf trainierbar sein."
        else:
            assert trainable == total, "Hier sollen alle Parameter trainierbar sein."
    print("TODO B ist korrekt.")
except NotImplementedError as e:
    print(f"{e} ist noch offen.")

### TODO C: Unterschiedliche Lernraten beim Finetuning

Beim Finetuning sind die Backbone-Gewichte bereits gut. Eine zu grosse Lernrate würde sie schnell «überschreiben» (*Catastrophic Forgetting*). Der neue Kopf startet dagegen zufällig und darf schneller lernen.

Ergänze `make_optimizer` so, dass beim Finetuning **zwei Parametergruppen** entstehen: das Backbone mit `LR_BACKBONE` und der Kopf mit `LR_HEAD`.

*Tipp:* `torch.optim.AdamW` akzeptiert statt einer Parameterliste auch eine Liste von Dictionaries, z. B. `[{"params": ..., "lr": ...}, {"params": ..., "lr": ...}]`.

In [ ]:
LR_HEAD = 1e-3       # Kopf (und Training von Grund auf)
LR_BACKBONE = 1e-4   # vortrainiertes Backbone beim Finetuning: 10x kleiner

def make_optimizer(model, strategy):
    if strategy == "scratch":
        return torch.optim.AdamW(model.parameters(), lr=LR_HEAD)
    if strategy == "feature_extraction":
        # nur die trainierbaren Parameter (also den Kopf) übergeben
        return torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR_HEAD)

    # TODO C: zwei Parametergruppen für das Finetuning
    head_params = list(model.fc.parameters())
    backbone_params = [p for name, p in model.named_parameters() if not name.startswith("fc.")]
    return torch.optim.AdamW([
        {"params": backbone_params, "lr": LR_BACKBONE},
        {"params": head_params, "lr": LR_HEAD},
    ])

In [ ]:
# Kontrollzelle C
try:
    opt = make_optimizer(build_model("finetuning"), "finetuning")
    lrs = sorted(g["lr"] for g in opt.param_groups)
    n_params = sum(len(g["params"]) for g in opt.param_groups)
    assert len(opt.param_groups) == 2, "Es braucht genau zwei Parametergruppen."
    assert np.allclose(lrs, [LR_BACKBONE, LR_HEAD]), "Die Lernraten der Gruppen stimmen nicht."
    assert n_params == len(list(build_model("finetuning").parameters())), "Es fehlen Parameter oder sind doppelt."
    print("TODO C ist korrekt. Lernraten:", lrs)
except NotImplementedError as e:
    print(f"{e} ist noch offen.")

## 4 Training (vorbereitet)

Die Trainingsfunktion kennst du im Prinzip aus Woche 2. Neu sind zwei Details:

1. **Bestes Modell behalten:** Nach jeder Epoche wird auf dem Validierungsset gemessen. Für den Test verwenden wir die Gewichte der besten Epoche.
2. **Stolperstein BatchNorm:** `requires_grad = False` stoppt nur die Gradienten. Die BatchNorm-Schichten aktualisieren im Trainingsmodus trotzdem ihre gespeicherten Mittelwerte und Varianzen. Bei Feature Extraction wollen wir das Backbone aber unverändert lassen. Darum schalten wir die BatchNorm-Schichten dort explizit in den `eval()`-Modus.

In [ ]:
BATCH_SIZE = 32
EPOCHS = {"100%": 20, "10%": 50}   # weniger Daten -> mehr Epochen, damit das Modell genügend Updates bekommt

def freeze_batchnorm(model):
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

@torch.no_grad()
def evaluate(model, x_uint8, y, normalize=True):
    model.eval()
    correct = 0
    for i in range(0, len(y), 256):
        xb = preprocess(x_uint8[i:i + 256].to(device), normalize)
        correct += (model(xb).argmax(1).cpu() == y[i:i + 256]).sum().item()
    return correct / len(y)

def train_model(model, strategy, train_split, epochs, optimizer=None, normalize=True, verbose=True):
    x_tr, y_tr = data[train_split]
    x_val, y_val = data["val"]
    optimizer = optimizer or make_optimizer(model, strategy)
    history, best_acc, best_state = [], -1.0, None
    for epoch in range(1, epochs + 1):
        model.train()
        if strategy == "feature_extraction":
            freeze_batchnorm(model)
        perm = torch.randperm(len(y_tr))
        running_loss = 0.0
        for i in range(0, len(y_tr), BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            xb = random_flip(preprocess(x_tr[idx].to(device), normalize))
            yb = y_tr[idx].to(device)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(idx)
        val_acc = evaluate(model, x_val, y_val, normalize)
        history.append({"epoch": epoch, "train_loss": running_loss / len(y_tr), "val_acc": val_acc})
        if val_acc > best_acc:
            best_acc, best_state = val_acc, copy.deepcopy(model.state_dict())
        if verbose and (epoch == 1 or epoch % 5 == 0 or epoch == epochs):
            print(f"  Epoche {epoch:2d}  Loss {history[-1]['train_loss']:.3f}  Val-Acc {val_acc:.3f}")
    model.load_state_dict(best_state)
    return history

## 5 Experiment

### Fragestellung
Wie stark verbessern vortrainierte ImageNet-Gewichte die Klassifikation von Satellitenbildern gegenüber dem Training von Grund auf? Und hängt die beste Strategie davon ab, wie viele gelabelte Trainingsbilder wir haben?

**Hypothese (vor dem Experiment notieren!):** Was erwartest du? Welche Strategie gewinnt bei 10 %, welche bei 100 %?

*Deine Hypothese:* …

### Experiment-Design
- **Variiert:** Strategie (3 Stufen) × Trainingsdatenmenge (100 % = 1'000 Bilder, 10 % = 100 Bilder), also 6 Läufe.
- **Konstant:** Architektur (ResNet-18), Splits, Vorverarbeitung, Augmentation, Batch Size, Seed, Optimizer (AdamW).
- **Metrik:** Accuracy auf dem Testset. Die Klassen sind balanciert, darum ist Accuracy hier sinnvoll. Ausgewertet wird jeweils das Modell der besten Validierungs-Epoche.
- **Einschränkung:** Pro Konfiguration gibt es nur einen Lauf (ein Seed). Wir können also keine Streuung angeben.

In [ ]:
def run(strategy, split):
    set_seed()
    model = build_model(strategy)
    t0 = time.time()
    hist = train_model(model, strategy, "train" if split == "100%" else "train_10", EPOCHS[split])
    duration = time.time() - t0
    test_acc = evaluate(model, *data["test"])
    total, trainable = count_params(model)
    return {"Strategie": strategy, "Daten": split, "Test-Acc": test_acc,
            "trainierbare Parameter": trainable, "Trainingszeit [s]": round(duration, 1),
            "history": hist}

results = []
for split in ["100%", "10%"]:
    for strategy in STRATEGIES:
        print(f"--- {strategy} | {split} Trainingsdaten ---")
        results.append(run(strategy, split))

### Resultate

In [ ]:
df = pd.DataFrame([{k: v for k, v in r.items() if k != "history"} for r in results])
df.pivot(index="Strategie", columns="Daten", values="Test-Acc").loc[STRATEGIES, ["10%", "100%"]].round(3)

In [ ]:
df[["Strategie", "Daten", "trainierbare Parameter", "Trainingszeit [s]"]]

In [ ]:
labels = {"scratch": "Von Grund auf", "feature_extraction": "Feature Extraction", "finetuning": "Finetuning"}
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, split in zip(axes, ["10%", "100%"]):
    for r in results:
        if r["Daten"] == split:
            h = pd.DataFrame(r["history"])
            ax.plot(h["epoch"], h["val_acc"], label=labels[r["Strategie"]])
    ax.set_title(f"{split} Trainingsdaten")
    ax.set_xlabel("Epoche")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Validierungs-Accuracy")
axes[0].legend()
plt.tight_layout()
plt.show()

### Interpretation

Beantworte die Fragen in 1–3 Sätzen. Die Musterantworten beziehen sich auf den Testlauf oben. Bei euch können die Zahlen leicht abweichen.

**1. Welche Strategie gewinnt bei 10 % und welche bei 100 % der Daten? Wie gross ist der Abstand zum Training von Grund auf?**
> *Musterantwort:* Beide vortrainierten Varianten liegen in beiden Szenarien deutlich vor dem Training von Grund auf. Im Testlauf: bei 10 % rund 42 % (von Grund auf) gegenüber 78 % (Feature Extraction) und 79 % (Finetuning), bei 100 % rund 61 % gegenüber 86 % und 92 %. Bei 100 % gewinnt Finetuning klar, bei 10 % liegen Feature Extraction und Finetuning nahe beieinander. Der Vorsprung gegenüber «von Grund auf» ist bei wenig Daten am grössten. Genau dort lohnt sich Transfer Learning also am meisten.

**2. Feature Extraction trainiert nur rund 0.05 % der Parameter. Wann lohnt sich der Mehraufwand für Finetuning?**
> *Musterantwort:* Feature Extraction ist schneller, braucht weniger Speicher und kann kaum überanpassen. Finetuning lohnt sich, wenn genügend Daten vorhanden sind und die Zieldomäne von den Vortrainingsdaten abweicht, weil das Backbone dann seine Merkmale an die neue Domäne anpassen kann. Bei sehr wenig Daten steigt beim Finetuning das Risiko, dass das Modell die Trainingsbilder auswendig lernt.

**3. Wo liegt EuroSAT in der Entscheidungsmatrix (Datenmenge × Ähnlichkeit zur Quelldomäne)?**
> *Musterantwort:* Satellitenbilder von oben sind keine typischen ImageNet-Fotos. Die Ähnlichkeit ist also nur mittel. Die einfachen Merkmale aus den frühen Schichten wie Kanten, Farben und Texturen sind aber trotzdem nützlich. Mit 100 bzw. 1'000 Bildern liegen wir klar im Bereich «wenig Daten». Das Resultat passt zur Matrix: Vortrainieren hilft immer, und mit etwas mehr Daten bringt Finetuning zusätzlich etwas.

**4. Was erwartest du, wenn du beim Finetuning auch für das Backbone `LR = 1e-3` verwendest?**
> *Musterantwort:* Die grossen Updates zu Beginn, wenn der Kopf noch zufällig ist und grosse Gradienten liefert, können die vortrainierten Merkmale zerstören (*Catastrophic Forgetting*). Typischerweise sinkt die Accuracy, und das Training wird instabiler, besonders bei 10 % der Daten. Probiere es aus.

**5. Ist der Vergleich fair? Was bräuchte es für eine belastbare Aussage?**
> *Musterantwort:* Fair ist, dass Splits, Architektur, Vorverarbeitung und Seed gleich sind. Es fehlen aber mehrere Läufe mit verschiedenen Seeds, um die Streuung abzuschätzen. Dazu kommt, dass die Hyperparameter (Lernrate, Epochen) nicht pro Strategie optimiert wurden. Das benachteiligt am ehesten das Training von Grund auf. Die grossen Abstände sind trotzdem glaubwürdig, kleine Unterschiede von 1–2 Prozentpunkten dagegen nicht.

**6. Ein Kunde hat 200 gelabelte Drohnenbilder von Dächern und möchte Schäden erkennen. Was empfiehlst du als ersten Schritt?**
> *Musterantwort:* Ich würde mit einem vortrainierten Modell und Feature Extraction als schneller Baseline starten und danach vorsichtig finetunen, mit kleiner Lernrate oder schrittweisem Auftauen. Ein Training von Grund auf ergibt mit 200 Bildern keinen Sinn. Ausserdem würde ich die Lizenz der vortrainierten Gewichte für den kommerziellen Einsatz prüfen und abklären, ob es ein Modell gibt, das bereits auf Luftbildern vortrainiert ist.

## 6 Bonus (falls du schneller bist)

### Bonus 1: Falsche Vorverarbeitung
Was passiert bei Feature Extraction, wenn du die ImageNet-Normalisierung weglässt? Das Backbone sieht dann Werte zwischen 0 und 1 statt der gewohnten normalisierten Werte.

In [ ]:
set_seed()
m_nonorm = build_model("feature_extraction")
train_model(m_nonorm, "feature_extraction", "train_10", EPOCHS["10%"], normalize=False, verbose=False)
acc_nonorm = evaluate(m_nonorm, *data["test"], normalize=False)
acc_norm = df.query("Strategie == 'feature_extraction' and Daten == '10%'")["Test-Acc"].item()
print(f"Feature Extraction, 10 %:  mit Normalisierung {acc_norm:.3f}  |  ohne Normalisierung {acc_nonorm:.3f}")

> *Musterantwort:* Ohne Normalisierung fällt die Accuracy tiefer aus, im Testlauf von rund 78 % auf 66 %. Der neue Kopf kann zwar einen Teil davon ausgleichen, das eingefrorene Backbone sieht aber eine ganz andere Verteilung der Eingabewerte als beim Vortraining. Wie gross der Unterschied ist, hängt vom Datensatz ab. In euren Projekten ist eine falsche oder fehlende Vorverarbeitung eine häufige stille Fehlerquelle: Der Code läuft, nur die Resultate sind schlechter.

### Bonus 2: Schrittweises Auftauen (*Gradual Unfreezing*)
Statt alles auf einmal zu trainieren, trainierst du zuerst nur den Kopf. Danach taust du schrittweise die obersten Blöcke auf (`layer4`, dann `layer3`). Die Blöcke nahe am Input bleiben eingefroren.

Implementiere drei Phasen mit je 10 Epochen auf den 10-%-Daten und vergleiche das Resultat mit der Tabelle oben.

In [ ]:
set_seed()
m_gu = build_model("feature_extraction")          # Start: alles eingefroren ausser dem Kopf
phases = [[], ["layer4"], ["layer4", "layer3"]]   # welche Blöcke in welcher Phase trainierbar sind
for unfrozen in phases:
    for name in unfrozen:
        for p in getattr(m_gu, name).parameters():
            p.requires_grad = True
    opt = torch.optim.AdamW([
        {"params": [p for n, p in m_gu.named_parameters() if p.requires_grad and not n.startswith("fc.")], "lr": LR_BACKBONE},
        {"params": m_gu.fc.parameters(), "lr": LR_HEAD},
    ])
    # "finetuning" als Strategie, damit BatchNorm in den aufgetauten Blöcken normal trainiert wird
    train_model(m_gu, "finetuning" if unfrozen else "feature_extraction", "train_10", 10, optimizer=opt, verbose=False)
    print(f"Phase mit {unfrozen or 'nur Kopf'}: trainierbar {count_params(m_gu)[1]:,}  Val-Acc {evaluate(m_gu, *data['val']):.3f}")
print(f"Test-Acc Gradual Unfreezing (10 %): {evaluate(m_gu, *data['test']):.3f}")

> *Musterantwort:* Schrittweises Auftauen ist ein Kompromiss zwischen Feature Extraction und vollem Finetuning. Im Testlauf erreichte es rund 80 % und lag damit leicht vor Feature Extraction (78 %) und vollem Finetuning (79 %). Bei wenig Daten liefert es oft ein vergleichbares oder leicht besseres Resultat als volles Finetuning, weil die allgemeinen Merkmale der frühen Schichten geschützt bleiben. Der Unterschied ist in diesem kleinen Experiment aber nicht zwingend belastbar (siehe Frage 5).

---
**Ausblick:** Statt ImageNet-CNNs kannst du auch grössere *Foundation Models* als Backbone verwenden, zum Beispiel DINOv2 (selbstüberwachtes Vortraining) oder den Bildteil von CLIP. Viele solche Modelle findest du in der Bibliothek `timm` oder auf dem Hugging Face Hub. Lies vor dem Einsatz immer die Model Card: Auf welchen Daten wurde das Modell vortrainiert, und welche Lizenz gilt?